# Generative Recipe Model — Train on Google Colab (Optimized Version)

This optimized notebook trains a language model to invent brand-new recipes. It includes fixes for memory crashes, robust file uploads, and faster preprocessing.

## Runtime
Runtime → Change runtime type → **T4 GPU** (or better) before running.

## 0. Install dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate kaggle openpyxl
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 1. Stage 1 — download the Food.com dataset

We use the **Food.com Recipes and Interactions** dataset on Kaggle.
1. Go to kaggle.com → Settings → API → **Create New Token** to download `kaggle.json`.
2. Run the cell below. (If you already uploaded it, it won't ask again).

In [ ]:
import os
from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        os.makedirs("/root/.kaggle", exist_ok=True)
        !mv kaggle.json /root/.kaggle/kaggle.json
        !chmod 600 /root/.kaggle/kaggle.json
        print("Kaggle token configured successfully.")
    else:
        print("Error: kaggle.json was not uploaded.")
else:
    print("Kaggle token already configured.")

In [ ]:
import os
if not os.path.exists("/content/foodcom/RAW_recipes.csv"):
    !kaggle datasets download -d shuyangli94/food-com-recipes-and-user-interactions -f RAW_recipes.csv -p /content/foodcom --force
    !unzip -o /content/foodcom/RAW_recipes.csv.zip -d /content/foodcom
else:
    print("Dataset already downloaded.")

## 2. Load and inspect Food.com data

In [ ]:
import pandas as pd
import ast

foodcom_df = pd.read_csv("/content/foodcom/RAW_recipes.csv")
print(f"Total raw recipes: {len(foodcom_df)}")
foodcom_df.head(2)

## 3. Efficient Preprocessing & Subsampling

To avoid running out of RAM, we filter the dataset *before* parsing the stringified lists.

In [ ]:
# Filter to keep recipes with a sane number of ingredients/steps and a title
mask = (
    foodcom_df["n_ingredients"].between(2, 20)
    & foodcom_df["n_steps"].between(2, 15)
    & foodcom_df["name"].str.len().fillna(0).gt(2)
)
clean_df = foodcom_df[mask].reset_index(drop=True)
print(f"Kept {len(clean_df)} / {len(foodcom_df)} recipes after filtering")

# Subsample FIRST to save memory and time parsing
SAMPLE_SIZE = 60000
if len(clean_df) > SAMPLE_SIZE:
    clean_df = clean_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

def safe_literal_eval(val):
    try:
        return ast.literal_eval(val)
    except:
        return []

# Safely parse lists only for the subsampled data
clean_df["tags_list"] = clean_df["tags"].apply(safe_literal_eval)
clean_df["ingredients_list"] = clean_df["ingredients"].apply(safe_literal_eval)
clean_df["steps_list"] = clean_df["steps"].apply(safe_literal_eval)

## 4. Curate a "mood" tag vocabulary

In [ ]:
from collections import Counter

CANDIDATE_MOOD_TAGS = [
    "sweet", "desserts", "spicy", "savory", "salty", "sour", "healthy",
    "low-calorie", "comfort-food", "quick", "easy", "vegetarian", "vegan",
    "breakfast", "lunch", "dinner", "snacks", "soups-stews", "seafood",
    "chicken", "beef", "pork", "chocolate", "fruit", "summer", "winter",
    "holiday-event", "kid-friendly", "main-dish", "side-dishes",
    "appetizers", "beverages", "spicy",
]

tag_counts = Counter()
for tags in clean_df["tags_list"]:
    tag_counts.update(tags)

KEPT_TAGS = sorted({t for t in CANDIDATE_MOOD_TAGS if tag_counts.get(t, 0) > 100})
print(f"Kept {len(KEPT_TAGS)} mood/taste tags.")

## 5. Build the training text format

In [ ]:
def format_foodcom_row(row):
    tags = [t for t in row.get("tags_list", []) if t in KEPT_TAGS][:6]
    lines = ["<|startofrecipe|>"]
    lines.append("tags: " + (", ".join(tags) if tags else "general"))
    lines.append("title: " + str(row["name"]).strip())
    lines.append("ingredients:")
    for ing in row.get("ingredients_list", []):
        lines.append("- " + str(ing).strip())
    lines.append("steps:")
    for i, step in enumerate(row.get("steps_list", []), 1):
        lines.append(f"{i}. " + str(step).strip())
    lines.append("<|endofrecipe|>")
    return "\n".join(lines)

clean_df["formatted_text"] = clean_df.apply(format_foodcom_row, axis=1)
print(clean_df["formatted_text"].iloc[0])

## 6. Train/validation split & Tokenizer setup

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import GPT2Tokenizer

train_texts, val_texts = train_test_split(
    clean_df["formatted_text"].tolist(), test_size=0.02, random_state=42
)
print(f"Train: {len(train_texts)}  Val: {len(val_texts)}")

BASE_MODEL = "distilgpt2"
tokenizer = GPT2Tokenizer.from_pretrained(BASE_MODEL)
tokenizer.add_special_tokens({
    "pad_token": "<|pad|>",
    "additional_special_tokens": ["<|startofrecipe|>", "<|endofrecipe|>"],
})

## 7. Efficient Tokenization using Hugging Face Datasets

In [ ]:
from datasets import Dataset

BLOCK_SIZE = 256

def tokenize_texts(texts):
    # Standard efficient HF map approach for block tokenization
    ds = Dataset.from_dict({"text": texts})
    def tokenize_function(examples):
        return tokenizer(examples["text"])
    
    tokenized_ds = ds.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])
    
    def group_texts(examples):
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
        result = {
            k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    return tokenized_ds.map(group_texts, batched=True, num_proc=4)

train_dataset_stage1 = tokenize_texts(train_texts)
val_dataset_stage1 = tokenize_texts(val_texts)
print("Tokenized Datasets:")
print(train_dataset_stage1)
print(val_dataset_stage1)

## 8. Load model and train — Stage 1 (Food.com)

In [ ]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

model = GPT2LMHeadModel.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))
model.to("cuda" if torch.cuda.is_available() else "cpu")

stage1_args = TrainingArguments(
    output_dir="/content/stage1_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16, # Increased batch size for efficiency
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_stage1 = Trainer(
    model=model,
    args=stage1_args,
    train_dataset=train_dataset_stage1,
    eval_dataset=val_dataset_stage1,
)

trainer_stage1.train()

In [ ]:
import math

eval_results = trainer_stage1.evaluate()
print(f"Stage 1 eval loss: {eval_results['eval_loss']:.4f}  perplexity: {math.exp(eval_results['eval_loss']):.2f}")

STAGE1_DIR = "/content/recipe_gpt2_stage1_foodcom"
trainer_stage1.save_model(STAGE1_DIR)
tokenizer.save_pretrained(STAGE1_DIR)
print("Saved Stage 1 model to", STAGE1_DIR)

### Sanity check — generate from the Stage 1 model before continuing

In [ ]:
def quick_generate(prompt, model, tokenizer, max_new_tokens=200):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.9,
        top_p=0.92,
        top_k=50,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=False)

print(quick_generate("<|startofrecipe|>\ntags: sweet, desserts\ntitle:", model, tokenizer))

## 9. Stage 2 — transfer to your Sri Lankan recipe dataset

Please upload `Sri_Lankan_Recipe_Database (1).xlsx` when prompted. The cell checks if it's already uploaded.

In [ ]:
import glob
from google.colab import files

xlsx_files = glob.glob("*.xlsx")
if not xlsx_files:
    print("Please upload Sri_Lankan_Recipe_Database (1).xlsx")
    uploaded = files.upload()
    xlsx_files = glob.glob("*.xlsx")

if xlsx_files:
    xlsx_path = xlsx_files[0]
    print("Using file:", xlsx_path)
    sl_df = pd.read_excel(xlsx_path, sheet_name="Recipes")
    print(sl_df.shape)
    display(sl_df.head(2))
else:
    print("Error: No .xlsx file found!")

### Preprocess Sri Lankan recipes
We safely format the rows and handle missing values.

In [ ]:
def format_srilankan_row(row):
    tag_fields = ["Category", "Food_Group", "Dietary_Tag", "Meal_Type", "Spice_Level"]
    tags = []
    for field in tag_fields:
        v = str(row.get(field, "")).strip().lower()
        if v and v != "nan" and v not in tags:
            tags.append(v)
            
    # Compute weight loss suitability
    cal = pd.to_numeric(row.get("Calories_kcal"), errors='coerce')
    if pd.notna(cal):
        tags.append("low-cal" if cal < 200 else "moderate" if cal < 400 else "high-cal")

    ingredients = [x.strip() for x in str(row.get("Key_Ingredients", "")).split(",") if x.strip()]

    lines = ["<|startofrecipe|>"]
    lines.append("tags: " + ", ".join(tags))
    lines.append("title: " + str(row.get("Recipe_Name", "")).strip())
    lines.append("ingredients:")
    for ing in ingredients:
        lines.append("- " + ing)
        
    if all(pd.notna(row.get(k)) for k in ["Calories_kcal", "Protein_g", "Carbs_g", "Fat_g", "Fiber_g"]):
        lines.append(
            "nutrition: {:.0f} kcal, {:.1f}g protein, {:.1f}g carbs, {:.1f}g fat, {:.1f}g fiber".format(
                row["Calories_kcal"], row["Protein_g"], row["Carbs_g"], row["Fat_g"], row["Fiber_g"]
            )
        )
        
    desc = str(row.get("Short_Description", "")).strip()
    if desc and desc.lower() != "nan":
        lines.append("description: " + desc)
    lines.append("<|endofrecipe|>")
    return "\n".join(lines)

sl_df["formatted_text"] = sl_df.apply(format_srilankan_row, axis=1)
print(sl_df["formatted_text"].iloc[0])

### Build the Stage 2 training set

In [ ]:
import random
random.seed(42)

def shuffled_tags_variant(row):
    text = format_srilankan_row(row)
    # Basic tag shuffling to create variants
    parts = text.split("\n")
    if len(parts) > 1 and parts[1].startswith("tags:"):
        tags = parts[1].replace("tags: ", "").split(", ")
        random.shuffle(tags)
        parts[1] = "tags: " + ", ".join(tags)
    return "\n".join(parts)

OVERSAMPLE_FACTOR = 4
sl_texts = []
for _ in range(OVERSAMPLE_FACTOR):
    sl_texts.extend(sl_df.apply(shuffled_tags_variant, axis=1).tolist())

REPLAY_SIZE = 3000
replay_texts = random.sample(train_texts, min(REPLAY_SIZE, len(train_texts)))

stage2_texts = sl_texts + replay_texts
random.shuffle(stage2_texts)
print(f"Stage 2 training examples: {len(stage2_texts)} "
      f"({len(sl_texts)} Sri Lankan + {len(replay_texts)} Food.com replay)")

stage2_train_texts, stage2_val_texts = train_test_split(stage2_texts, test_size=0.05, random_state=42)
train_dataset_stage2 = tokenize_texts(stage2_train_texts)
val_dataset_stage2 = tokenize_texts(stage2_val_texts)


### Continue training from the Stage 1 checkpoint

In [ ]:
stage2_model = GPT2LMHeadModel.from_pretrained(STAGE1_DIR)
stage2_model.to("cuda" if torch.cuda.is_available() else "cpu")

stage2_args = TrainingArguments(
    output_dir="/content/stage2_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    learning_rate=3e-5,
    warmup_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_stage2 = Trainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=train_dataset_stage2,
    eval_dataset=val_dataset_stage2,
)

trainer_stage2.train()

In [ ]:
eval_results_2 = trainer_stage2.evaluate()
print(f"Stage 2 eval loss: {eval_results_2['eval_loss']:.4f}  perplexity: {math.exp(eval_results_2['eval_loss']):.2f}")

FINAL_DIR = "/content/recipe_gpt2_final_srilankan"
trainer_stage2.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Saved final model to", FINAL_DIR)

## 10. Mood-conditioned generation

Robust parsing logic for generated text.

In [ ]:
MOOD_KEYWORDS = {
    "sweet": ["sweet", "dessert"], "spicy": ["spicy", "hot"],
    "savory": ["savory"], "healthy": ["healthy", "low-cal"],
    "diet": ["low-cal"], "quick": ["quick", "easy"],
    "vegetarian": ["vegetarian"], "vegan": ["vegan"],
    "breakfast": ["breakfast"], "dinner": ["dinner"],
}

def build_prompt(mood_text):
    mood_lower = mood_text.lower()
    matched = []
    for kw, tags in MOOD_KEYWORDS.items():
        if kw in mood_lower:
            for t in tags:
                if t not in matched:
                    matched.append(t)
    if not matched:
        matched = [mood_lower.strip()]
    return f"<|startofrecipe|>\ntags: {', '.join(matched)}\ntitle:"

import re

def parse_generated_recipe(text):
    text = text.split("<|endofrecipe|>")[0].replace("<|startofrecipe|>", "").strip()
    
    def extract(label, next_labels):
        pattern = rf"{label}:\s*(.*?)(?=(?:{'|'.join(next_labels)}):|$)"
        m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        return m.group(1).strip() if m else ""

    labels_order = ["tags", "title", "ingredients", "steps", "nutrition", "description"]
    result = {}
    for i, label in enumerate(labels_order):
        result[label] = extract(label, labels_order[i + 1:])
    return result

def generate_recipe(mood_text, model, tokenizer, num_return_sequences=1):
    prompt = build_prompt(mood_text)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(
        input_ids,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.9,
        top_p=0.92,
        top_k=50,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    return [parse_generated_recipe(tokenizer.decode(o, skip_special_tokens=False)) for o in outputs]

def print_recipe(r):
    for k, v in r.items():
        if v:
            print(f"{k.upper()}:\n{v}")
    print("-" * 60)

In [ ]:
for mood in ["something sweet", "spicy and healthy", "a quick vegetarian dinner"]:
    print(f"=== Mood: {mood!r} ===")
    for r in generate_recipe(mood, stage2_model, tokenizer, num_return_sequences=1):
        print_recipe(r)

## 11. Save the final model

In [ ]:
!zip -r /content/recipe_gpt2_final_srilankan.zip {FINAL_DIR}
from google.colab import files
try:
    files.download("/content/recipe_gpt2_final_srilankan.zip")
    print("Downloading model zip...")
except:
    print("Download failed or running in non-interactive environment.")